# Residual structure — which input space organises the ConvCNP residuals?

Standalone companion to `cross_folder_analysis.ipynb`, focused on the spatial /
descriptor structure of the per-station residuals for the five single-region 14y
snapshots. These sections were lifted out of the main notebook (originally its
sections 10, 11 and 12) and renumbered here:

1. **Spatial residual maps** — baseline vs TESSERA, per region x variable.
2. **Geographic vs TESSERA-latent neighbour roughness** — the two-space structure test.
3. **Which input space organises the residuals?** — the full four-descriptor
   decomposition (geographic / elevation+mTPI / ERA5-static / TESSERA lat16), each
   scored by the same neighbourhood **structure score** as §2 and shown as a
   which-space-wins bar + per-model grouped bars. Reuses the descriptor-space
   recipe from `scripts/analysis/norway_descriptor_spaces.py`.

Run the two setup cells first, then Sections 1 -> 2 -> 3 in order (3 reuses helpers from 1-2).

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Local helpers — YAML-driven loader, summary tables, shortlists.
sys.path.insert(0, str(Path.cwd()))
import importlib

import _helpers

importlib.reload(_helpers)  # convenient for iterative edits to _helpers.py

warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update(
    {
        "figure.figsize": (12, 6),
        "figure.dpi": 120,
        "font.size": 11,
        "axes.titlesize": 13,
        "axes.labelsize": 12,
        "figure.constrained_layout.use": True,
    }
)

In [ ]:
# =============================================================================
# TESSERA generation selector — pick which latents generation the WHOLE
# notebook analyses as "the TESSERA arm" (model residuals AND the TESSERA
# latent descriptor space). The no-TESSERA baselines are latents-independent
# and always come from the base region folders. Selected-generation runs are
# mapped onto the canonical v1 folder/stem keys at load time (see the setup
# cell below), so every downstream cell follows this choice transparently;
# a row's run_dir always records the true run that produced it.
# =============================================================================

# --- current main: TESSERA v2 "1B-M", 2017 embeddings, crop64_lat16_auxon ---
TESS_GEN = "1B-M 2017"
TESS_FOLDER_SUFFIX = "_tessera_1B-M_2017"
TESS_STEM = {
    "t2m": "t2m_snap_vae_crop64_lat16_auxon_concat_mtpi",
    "wind": "wind_truncnormal_snap_vae_crop64_lat16_auxon_concat_mtpi",
}

# --- previous main: TESSERA v1 16-d latents (uncomment to switch back) ------
# TESS_GEN = "v1 lat16"
# TESS_FOLDER_SUFFIX = ""
# TESS_STEM = {"t2m":  "t2m_snap_vae_lat16_concat_with_elev_mtpi_no_static_wd",
#              "wind": "wind_truncnormal_snap_vae_lat16_concat_with_elev_mtpi_no_static_wd"}

# Canonical (v1) stems — the keys downstream cells select on (SR_MODELS).
_CANON_STEM = {
    "t2m": "t2m_snap_vae_lat16_concat_with_elev_mtpi_no_static_wd",
    "wind": "wind_truncnormal_snap_vae_lat16_concat_with_elev_mtpi_no_static_wd",
}

print(f"TESSERA generation: {TESS_GEN}  (folder suffix: '{TESS_FOLDER_SUFFIX}')")

In [ ]:
# Restricted to the five single-region 14y snapshots that Sections 1-3 analyse.
FOLDERS = [
    "snapshot_14y_eu",
    "snapshot_14y_us",
    "snapshot_14y_east_asia",
    "snapshot_14y_australia",
    "snapshot_14y_southern_africa",
]
SEEDS = [42, 123, 456]
df = _helpers.load_all_results(folders=FOLDERS, seeds=SEEDS)

if TESS_FOLDER_SUFFIX:
    # Swap the TESSERA arm for the selected generation: drop the v1 TESSERA
    # rows and map the generation's runs onto the canonical folder/stem keys.
    _df_gen = _helpers.load_all_results(
        folders=[f + TESS_FOLDER_SUFFIX for f in FOLDERS], seeds=SEEDS
    )
    _df_gen = _df_gen[_df_gen["experiment"].isin(TESS_STEM.values())].copy()
    _df_gen["experiment"] = _df_gen["experiment"].map(
        {TESS_STEM[v]: _CANON_STEM[v] for v in TESS_STEM}
    )
    _df_gen["source_folder"] = _df_gen["source_folder"].str.replace(
        TESS_FOLDER_SUFFIX, "", regex=False
    )
    df = pd.concat(
        [df[~df["experiment"].isin(_CANON_STEM.values())], _df_gen], ignore_index=True
    )
    print(
        f"TESSERA arm swapped to {TESS_GEN}: {len(_df_gen)} runs mapped "
        "onto the canonical stems"
    )

print(
    f"Loaded {len(df)} run results across {df['experiment'].nunique()} "
    f"experiments and {df['source_folder'].nunique()} folders"
)

## 1. Spatial residual maps — baseline vs TESSERA (all regions × {t2m, wind})

Loops over every single-region 14y folder and both variables. For each, plots
per-station residuals on a lon/lat map for a trained **No-TESSERA** ConvCNP
baseline and the matched **TESSERA lat16 + concat** model, side by side on one
symmetric diverging colour scale, with a third panel showing the per-station
change (TESSERA − baseline).

**What `bias` means here:** the per-station **mean signed residual**
(predicted − observed), time-averaged over that station's test timestamps —
the standard term for systematic over/under-prediction (positive = the model
runs warm/high there, negative = cold/low). It is *not* a single prediction's
error: one timestamp's `pred − obs` is a *residual*, and its station-mean is
the *bias*. `mae`/`rmse` (switch via the `SR_METRIC` knob) instead measure
error *magnitude* and never cancel. Bias is the most useful view for this
question because spatial structure in systematic error (warm/cold patches) is
exactly where a "smoother with TESSERA" effect appears.

The question: does TESSERA produce a *spatially smoother* error field —
neighbouring stations erring together rather than a salt-and-pepper pattern?
For each figure we record a k-nearest-neighbour **roughness** statistic
(mean |residual − mean residual of its k neighbours|; lower ⇒ smoother) and
print a cross-region summary table at the end (negative Δ% ⇒ TESSERA smoother).

Per-station residuals are read from each run's `test_station_errors.npz`
(written by `evaluate.py`), averaged across seeds per station, masking stations
with zero test predictions. Edit `SR_LOSS` / `SR_METRIC` / `SR_KNN` /
`SR_REGION_FOLDERS` / `SR_VARS` at the top of the code cell.

In [ ]:
# =============================================================================
# Spatial residual maps — baseline vs TESSERA, every region × {t2m, wind}.
#
# For each single-region 14y folder and each variable, plots per-station
# residuals on a lon/lat map for a trained No-TESSERA ConvCNP baseline and the
# matched TESSERA lat16 + concat model, side by side on one symmetric diverging
# colour scale, plus a third panel for the per-station change (TESSERA − base).
#
# Default metric is `bias` = the per-station MEAN SIGNED residual
# (predicted − observed), time-averaged over that station's test timestamps —
# the standard term for systematic over/under-prediction. (A single timestamp's
# pred − obs is a *residual*; its station-mean is the *bias*. `mae`/`rmse` are
# the magnitude of error and never cancel.) Bias is the most useful view here
# because spatial structure in systematic error (warm/cold patches) is exactly
# where "smoother with TESSERA" shows up.
#
# Alongside each figure we record a k-nearest-neighbour roughness statistic
# (mean |residual_i − mean residual of its k neighbours|); lower ⇒ the error
# varies more smoothly across space. A summary table is printed at the end.
#
# Per-station residuals come from each run's test_station_errors.npz (written
# by evaluate.py); values are averaged across seeds per station, masking
# stations with zero test predictions in a given run.
# =============================================================================

# ---- knobs ----
SR_LOSS = "nll"  # training loss applied to baseline & tessera
SR_METRIC = "bias"  # "bias" (signed) | "mae" | "rmse"
SR_KNN = 8  # neighbours for the roughness statistic

# Single-region 14y snapshot folders -> pretty region name (same mapping as
# the §4 plotting cell in cross_folder_analysis.ipynb; built explicitly because _folder_to_region maps
# 'australia' -> 'us' via substring).
SR_REGION_FOLDERS = {
    "snapshot_14y_eu": "Europe",
    "snapshot_14y_us": "United States",
    "snapshot_14y_east_asia": "East Asia",
    "snapshot_14y_australia": "Australia",
    "snapshot_14y_southern_africa": "Southern Africa",
}
# (variable, head distribution, unit).
SR_VARS = [("t2m", "gaussian", "°C"), ("wind", "truncated_normal", "m s$^{-1}$")]

_clf = _helpers._augment_df_with_classification(df)  # idempotent


# The two model variants we carry forward for ALL residual-structure analysis
# (FIXED choices, never a best-of-sweep pick). Exact experiment names per
# variable; each has three seeds. Both share the recipe
# bilinear interpolation, +elevation, +mTPI, and differ only in the SURFACE
# conditioning:
#   TESSERA  : lat16 direct-concat embedding, NO ERA5 static fields
#   Baseline : no TESSERA, WITH ERA5 static fields
# So §3's ERA5-static descriptor is the baseline's own surface input and the
# TESSERA-lat16 descriptor is tessera's — the two are a direct contrast.
SR_MODELS = {
    "t2m": {
        "baseline": "t2m_snap_bilinear_baseline_mtpi_wd",
        "tessera": "t2m_snap_vae_lat16_concat_with_elev_mtpi_no_static_wd",
    },
    "wind": {
        "baseline": "wind_truncnormal_snap_bilinear_baseline_mtpi_wd",
        "tessera": "wind_truncnormal_snap_vae_lat16_concat_with_elev_mtpi_no_static_wd",
    },
}


def _sr_select(folder, var, dist=None, loss=None):
    """(baseline_rows, tessera_rows): all seed rows for the two FIXED model
    variants in SR_MODELS, matched by exact experiment name — the specific
    baseline/TESSERA pair we carry forward everywhere, never a best-of-sweep
    pick. `dist`/`loss` are accepted for call-site compatibility but unused: the
    experiment name already pins the distribution head and (nll) loss."""
    pool = _clf[_clf["source_folder"] == folder]
    names = SR_MODELS.get(var, {})
    base = pool[pool["experiment"] == names.get("baseline", "")]
    tess = pool[pool["experiment"] == names.get("tessera", "")]
    return base, tess


def _sr_load(rows, var, metric):
    """station_id -> (lat, lon, mean metric across seeds). Skips count==0."""
    acc = {}  # sid -> [lat, lon, sum, n]
    mkey, ckey = f"{var}_station_{metric}", f"{var}_station_count"
    for run_dir in rows["run_dir"]:
        npz = Path(run_dir) / "test_station_errors.npz"
        if not npz.exists():
            continue
        d = np.load(npz, allow_pickle=True)
        if mkey not in d.files:
            continue
        sids, lats, lons = d["station_ids"], d["station_lats"], d["station_lons"]
        vals, cnt = d[mkey], d[ckey]
        for i, sid in enumerate(sids):
            if cnt[i] <= 0:
                continue
            e = acc.setdefault(str(sid), [float(lats[i]), float(lons[i]), 0.0, 0])
            e[2] += float(vals[i])
            e[3] += 1
    return {s: (e[0], e[1], e[2] / e[3]) for s, e in acc.items() if e[3] > 0}


def _sr_roughness(lat, lon, val, k):
    """Mean |val_i − mean(val over k nearest neighbours)|. Lower = smoother."""
    n = len(val)
    k = min(k, n - 1)
    if k < 1:
        return np.nan
    mlat = np.radians(lat.mean())
    x, y = lon * np.cos(mlat), lat  # equirectangular degrees
    d2 = (x[:, None] - x[None, :]) ** 2 + (y[:, None] - y[None, :]) ** 2
    np.fill_diagonal(d2, np.inf)
    nn = np.argsort(d2, axis=1)[:, :k]
    return float(np.mean(np.abs(val - val[nn].mean(axis=1))))


def _sr_plot(folder, region, var, dist, unit, loss, metric, knn, summary):
    base_rows, tess_rows = _sr_select(folder, var, dist, loss)
    if base_rows.empty or tess_rows.empty:
        missing = " & ".join(
            [
                s
                for s, e in (
                    ("baseline", base_rows.empty),
                    ("tessera", tess_rows.empty),
                )
                if e
            ]
        )
        print(f"[skip] {region} · {var}: no {missing} run ({loss} loss)")
        return
    base = _sr_load(base_rows, var, metric)
    tess = _sr_load(tess_rows, var, metric)
    shared = sorted(set(base) & set(tess))
    if not shared:
        print(f"[skip] {region} · {var}: no overlapping stations")
        return
    lat = np.array([base[s][0] for s in shared])
    lon = np.array([base[s][1] for s in shared])
    vb = np.array([base[s][2] for s in shared])
    vt = np.array([tess[s][2] for s in shared])
    vd = vt - vb
    rb = _sr_roughness(lat, lon, vb, knn)
    rt = _sr_roughness(lat, lon, vt, knn)
    summary.append(
        dict(
            region=region,
            variable=var,
            n_stations=len(shared),
            rough_base=rb,
            rough_tessera=rt,
            rough_delta_pct=(100 * (rt - rb) / rb) if rb else np.nan,
        )
    )

    signed = metric == "bias"
    if signed:
        lim = np.nanpercentile(np.abs(np.concatenate([vb, vt])), 98)
        cm, vmin, vmax = "RdBu_r", -lim, lim
    else:
        cm, vmin, vmax = "viridis", 0.0, np.nanpercentile(np.concatenate([vb, vt]), 98)
    dlim = np.nanpercentile(np.abs(vd), 98) or 1.0
    panels = [
        ("ConvCNP (without TESSERA)", vb, cm, vmin, vmax),
        ("ConvCNP with TESSERA", vt, cm, vmin, vmax),
        ("Δ  (with − without TESSERA)", vd, "RdBu_r", -dlim, dlim),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(20, 5.5))
    for ax, (title, v, cmv, lo, hi) in zip(axes, panels, strict=False):
        sc = ax.scatter(
            lon,
            lat,
            c=v,
            cmap=cmv,
            vmin=lo,
            vmax=hi,
            s=18,
            edgecolor="k",
            linewidth=0.15,
        )
        ax.set_title(title, fontsize=11)
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")
        ax.set_aspect(1.0 / np.cos(np.radians(lat.mean())))
        ax.grid(ls=":", alpha=0.4)
        ax.set_axisbelow(True)
        cb = fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
        cb.set_label(f"{metric} ({unit})")
    fig.suptitle(
        f"{region} — per-station {metric} · {var}/{dist} · {loss.upper()} loss   "
        f"(n={len(shared)}, roughness {rb:.3f}→{rt:.3f}, {100 * (rt - rb) / rb:+.1f}%)",
        fontsize=12,
        y=1.04,
    )
    plt.show()


SR_SUMMARY = []
for _folder, _region in SR_REGION_FOLDERS.items():
    for _var, _dist, _unit in SR_VARS:
        _sr_plot(
            _folder, _region, _var, _dist, _unit, SR_LOSS, SR_METRIC, SR_KNN, SR_SUMMARY
        )

print(
    f"\nk-NN (k={SR_KNN}) {SR_METRIC} roughness summary — lower = spatially smoother; "
    f"negative Δ% = TESSERA smoother than baseline:"
)
display(pd.DataFrame(SR_SUMMARY).round(4))

## 2. Stronger test — latent-space-neighbour roughness

Section 1's roughness uses **geographic** neighbours, which conflates
"physically similar" with "geographically near": two adjacent stations can be
physically very different (ridge vs valley), so a *correct* model looks rough
right where land cover changes sharply. This cell recomputes the same statistic
but defines neighbours in **TESSERA latent space** (k nearest in the z-scored
lat16 embedding).

If the latents encode the physical land-surface properties that drive the
baseline's local errors, then stations close in latent space are physically
alike and should share similar systematic bias — so the residual field should
be **smooth when ordered by latent similarity**. The strong claim therefore
predicts, *for the baseline*, `latent_score > geo_score`: scatter that looks
random in space is actually organised by the physical descriptor the latents
carry.

Each roughness is normalised against a **random-neighbour** reference (the
"no-structure" ceiling, averaged over many deterministic draws so it is stable
and reproducible — see `_sr_rough_random`) into a
`structure score = 1 − rough/rough_random`
(≈0 = that ordering explains no local bias structure; →1 = highly coherent).
The cell prints a per-(region, variable) table and a grouped-bar figure
comparing geo- vs latent-neighbour structure for both the baseline and TESSERA.
Reuses the `SR_*` knobs and `_sr_select` / `_sr_load` helpers from section 1.

In [ ]:
# =============================================================================
# Stronger test — latent-space-neighbour roughness vs geographic-neighbour.
#
# The geographic roughness in section 1 conflates "physically similar" with
# "geographically near": two adjacent stations can be physically very different
# (ridge vs valley), so a correct model looks rough exactly where land cover
# changes sharply. This cell recomputes the same roughness statistic but with
# neighbours defined in TESSERA LATENT SPACE (k nearest in the z-scored lat16
# embedding) instead of lat/lon.
#
# Reasoning: if TESSERA latents encode the physical land-surface properties that
# drive the baseline's local errors, then stations close in latent space are
# physically alike and should share similar systematic bias — i.e. the residual
# field should be SMOOTH when ordered by latent similarity. So the strong claim
# predicts, for the BASELINE, latent-roughness < geographic-roughness: the
# scatter that looks random in space is actually organised by the physical
# descriptor the latents carry.
#
# Each roughness is normalised against a random-neighbour reference (same k, but
# neighbours drawn at random — the "no spatial/physical structure" ceiling)
# AVERAGED over many deterministic draws (see _sr_rough_random) into a
# STRUCTURE SCORE = 1 − rough / rough_random:
#     ~0  -> that neighbour ordering explains no local structure in the bias
#     ->1 -> bias is highly coherent across that notion of "nearby"
# Comparing the baseline's geo-score vs latent-score is the test; tessera scores
# are shown alongside. Uses the same SR_LOSS / SR_METRIC / SR_KNN / regions /
# vars knobs and the _sr_select / _sr_load helpers from section 1.
# =============================================================================

import json

_SR_LATENT_CACHE = {}


def _sr_latents(config_path):
    """station_id -> latent vector, from a run's vae_latents_path + station csv."""
    cfg = json.load(open(config_path))
    lp, cp = cfg.get("vae_latents_path"), cfg.get("vae_latents_station_csv")
    if not lp or not cp:
        return None
    # The config records the HPC path the run was trained with; map it onto
    # the current data root.
    from tessera_downscaling.paths import resolve

    lp, cp = resolve(lp), resolve(cp)
    if not lp.exists() or not cp.exists():
        return None
    key = (lp, cp)
    if key not in _SR_LATENT_CACHE:
        arr = np.load(lp)
        ids = pd.read_csv(cp)["station_id"].astype(str).values
        _SR_LATENT_CACHE[key] = {str(s): arr[i] for i, s in enumerate(ids)}
    return _SR_LATENT_CACHE[key]


def _sr_nn_idx(coords, k):
    """k nearest-neighbour indices (Euclidean) for an (n, d) coordinate array."""
    k = min(k, len(coords) - 1)
    d2 = ((coords[:, None, :] - coords[None, :, :]) ** 2).sum(-1)
    np.fill_diagonal(d2, np.inf)
    return np.argsort(d2, axis=1)[:, :k]


def _sr_rough_idx(val, nn):
    """Mean |val_i − mean(val over its neighbour set)|."""
    return float(np.mean(np.abs(val - val[nn].mean(axis=1))))


def _sr_rough_random(val, k, seed=0):
    """No-structure reference: roughness under RANDOM neighbour assignment,
    AVERAGED over many deterministic draws.

    A single random draw is far too noisy at small n (its std is ~4% of the
    roughness for n~40), which let the structure score's SIGN depend on the draw
    and made §2 and §3 disagree for the *same* descriptor space — they drew
    different references from a shared, advancing RNG. Averaging with a fixed
    seed pins the reference down and makes it identical wherever it is called, so
    the two sections now agree exactly. The draw count scales down with n (large
    n is already low-variance but costlier per draw); argpartition gives the k
    smallest without a full sort (the reference is an order-invariant set-mean).
    """
    n = len(val)
    k = min(k, n - 1)
    n_draws = int(np.clip(30000 // max(n, 1), 60, 500))
    rng = np.random.default_rng(seed)
    acc = 0.0
    for _ in range(n_draws):
        R = rng.random((n, n))
        np.fill_diagonal(R, np.inf)
        nn = np.argpartition(R, k, axis=1)[:, :k]
        acc += float(np.mean(np.abs(val - val[nn].mean(axis=1))))
    return acc / n_draws


def _sr_latent_test(folder, region, var, dist, loss, metric, knn, rows_out):
    base_rows, tess_rows = _sr_select(folder, var, dist, loss)
    if base_rows.empty or tess_rows.empty:
        print(f"[skip] {region} · {var}: missing baseline/tessera run ({loss} loss)")
        return
    lut = _sr_latents(Path(tess_rows["run_dir"].iloc[0]) / "config.json")
    if lut is None:
        print(f"[skip] {region} · {var}: no latents recorded in tessera run config")
        return
    base = _sr_load(base_rows, var, metric)
    tess = _sr_load(tess_rows, var, metric)
    shared = sorted(set(base) & set(tess) & set(lut))
    if len(shared) < knn + 2:
        print(f"[skip] {region} · {var}: only {len(shared)} stations with bias+latents")
        return

    lat = np.array([base[s][0] for s in shared])
    lon = np.array([base[s][1] for s in shared])
    vb = np.array([base[s][2] for s in shared])
    vt = np.array([tess[s][2] for s in shared])
    Lz = np.array([lut[s] for s in shared], dtype=float)
    Lz = (Lz - Lz.mean(0)) / (Lz.std(0) + 1e-8)  # per-feature z-score
    geo = np.column_stack([lon * np.cos(np.radians(lat.mean())), lat])

    gnn = _sr_nn_idx(geo, knn)
    lnn = _sr_nn_idx(Lz, knn)

    def scores(v):
        rg, rl = _sr_rough_idx(v, gnn), _sr_rough_idx(v, lnn)
        rr = _sr_rough_random(v, knn)  # deterministic, averaged reference
        return (
            rg,
            rl,
            rr,
            1 - rg / rr,
            1 - rl / rr,
        )  # roughs + geo/latent structure scores

    bg, bl, br, bgs, bls = scores(vb)
    tg, tl, tr, tgs, tls = scores(vt)
    rows_out.append(
        dict(
            region=region,
            variable=var,
            n=len(shared),
            base_geo=bg,
            base_latent=bl,
            base_rand=br,
            tess_geo=tg,
            tess_latent=tl,
            tess_rand=tr,
            base_geo_score=bgs,
            base_latent_score=bls,
            tess_geo_score=tgs,
            tess_latent_score=tls,
            latent_beats_geo=bls > bgs,
        )
    )


SR_LATENT_ROWS = []
for _folder, _region in SR_REGION_FOLDERS.items():
    for _var, _dist, _unit in SR_VARS:
        _sr_latent_test(
            _folder, _region, _var, _dist, SR_LOSS, SR_METRIC, SR_KNN, SR_LATENT_ROWS
        )

sr_lat_df = pd.DataFrame(SR_LATENT_ROWS)
print(f"\nNeighbour-space {SR_METRIC} roughness ({SR_LOSS.upper()} loss, k={SR_KNN}).")
print(
    "structure score = 1 − roughness/random_roughness  (0 = no structure, ->1 = coherent)."
)
print(
    "STRONG TEST: for the baseline, does latent-space ordering explain more "
    "local bias structure than geography (base_latent_score > base_geo_score)?"
)
display(sr_lat_df.round(4))

# ---- grouped-bar panels by region: baseline and TESSERA shown separately ----
# Two figures (one per model) so the geo-vs-latent contrast reads cleanly within
# a model. Shared y-limits across both figures keep baseline and TESSERA
# magnitudes directly comparable.
if not sr_lat_df.empty:
    model_groups = [
        (
            "ConvCNP (without TESSERA)",
            [
                ("base_geo_score", "geo nbrs", "#9ecae1"),
                ("base_latent_score", "TESSERA nbrs", "#3182bd"),
            ],
        ),
        (
            "ConvCNP with TESSERA",
            [
                ("tess_geo_score", "geo nbrs", "#fdae6b"),
                ("tess_latent_score", "TESSERA nbrs", "#e6550d"),
            ],
        ),
    ]
    _score_cols = [
        "base_geo_score",
        "base_latent_score",
        "tess_geo_score",
        "tess_latent_score",
    ]
    _ymin = min(0.0, float(sr_lat_df[_score_cols].min().min()))
    _ymax = float(sr_lat_df[_score_cols].max().max())
    _pad = 0.06 * (_ymax - _ymin if _ymax > _ymin else 1.0)
    _ylim = (_ymin - _pad, _ymax + _pad)

    for _model_name, _bars in model_groups:
        fig, axes = plt.subplots(
            1, len(SR_VARS), figsize=(8 * len(SR_VARS), 5), squeeze=False
        )
        for ax, (var, _dist, _unit) in zip(axes[0], SR_VARS, strict=False):
            sub = sr_lat_df[sr_lat_df["variable"] == var].set_index("region")
            regs = [r for r in SR_REGION_FOLDERS.values() if r in sub.index]
            x = np.arange(len(regs))
            w = 0.8 / len(_bars)
            for j, (col, label, c) in enumerate(_bars):
                off = (j - (len(_bars) - 1) / 2) * w
                ax.bar(
                    x + off,
                    sub.loc[regs, col].values,
                    w,
                    color=c,
                    edgecolor="black",
                    linewidth=0.3,
                    label=label,
                )
            ax.axhline(0, color="k", lw=0.8)
            ax.set_xticks(x)
            ax.set_xticklabels(regs, rotation=25, ha="right", fontsize=9)
            ax.set_ylabel(f"{SR_METRIC} structure score (1 − rough/random)")
            ax.set_title(
                f"{var} — local bias coherence by neighbour space", fontsize=11
            )
            ax.set_ylim(*_ylim)
            ax.grid(axis="y", ls=":", alpha=0.5)
            ax.set_axisbelow(True)
        axes[0][0].legend(fontsize=8, framealpha=0.9, loc="best")
        fig.suptitle(
            f"{_model_name}: geographic vs TESSERA-latent neighbour structure "
            f"of per-station {SR_METRIC}",
            fontsize=13,
            y=1.08,
        )
        plt.show()

## 3. Which input space organises the residuals? — four-descriptor decomposition

Section 2 pitted **geographic** against **TESSERA-latent** neighbours. This
section widens that test to the full set of per-station descriptor spaces the
question is really about, porting the recipe from
`scripts/analysis/norway_descriptor_spaces.py`:

1. **geographic** — (lat, lon), equirectangular (physically nearest stations),
   exactly as §2.
2. **elevation + mTPI** — (elevation, Δelevation, mTPI): a per-station physical
   descriptor **common to both** models' inputs (both set `include_elevation=True`
   and use the mTPI variant).
3. **ERA5-static (interp)** — the 13 coarse ERA5 static grids bilinearly
   interpolated to each station (clipped to grid bounds, no extrapolation). This
   is the **baseline's own surface conditioning** — the fixed baseline sets
   `include_static_fields=True`, whereas TESSERA drops the static fields
   (`no_static_fields=True`) and conditions on the embedding instead. We use the
   raw interpolated fields, **not** the CNN-encoded grid latent — the encoded
   latent is dominated by the transient weather flowing through the same CNN
   rather than persistent surface character (see the norway-script rationale).
4. **TESSERA lat16** — the 16-d land-surface embedding: **TESSERA's** surface
   conditioning (the baseline has no latents), exactly as §2.

For each space we z-score its features (geography keeps physical equirectangular
distance), take the `SR_KNN` nearest neighbours, and score the per-station
residual field with the same **structure score = 1 − rough/rough_random** as §2
(≈0 = that space explains no local residual structure; →1 = residuals are highly
coherent across that notion of "nearby"). The random-neighbour reference
`rough_random` is the shared, **deterministic + draw-averaged** `_sr_rough_random`
from §2, so the geographic and TESSERA-latent scores reported here match §2's
values exactly. The question — *which input space organises the residuals?* — is
answered by which space gives the **baseline** the highest structure score. The
sharpest head-to-head is **ERA5-static (the baseline's own surface input) vs
TESSERA lat16 (tessera's)**: if the baseline's residuals organise better under
lat16 than under the ERA5-static fields it actually conditions on, the embedding
captures surface structure driving those errors that the static fields miss —
which is what motivates conditioning on TESSERA.

The per-station elevation and ERA5-static descriptors come from the shared
`dataset_timestamp_global` (`stations.csv` + per-region `static_fields.npy`);
everything else reuses the `SR_*` knobs and the
`_sr_select / _sr_load / _sr_latents / _sr_nn_idx / _sr_rough_idx / _sr_rough_random`
helpers from Sections 1-2.

**Reading a bar depends on whether the model saw that space.** For a space the
model **ingested** (geographic and elevation+mTPI for both; ERA5-static for the
baseline; lat16 for TESSERA) the score is *unexploited residual* — structure the
model had the information to remove but didn't, so **lower is better**. For the
one space each model is **blind to** (lat16 for the baseline, ERA5-static for
TESSERA) the score is *unused signal* — surface-organised error along an axis it
never saw, so **higher = a bigger missed opportunity**. Because these two
readings are not commensurate, **we never average a space across the two
models**; the headline keeps them in separate panels, and every figure **hatches**
the blind space (solid = ingested / residual, hatched = blind / unused signal).

**What TESSERA absorbs (headline panel 3).** Δ = baseline − TESSERA structure per
space; positive = TESSERA flattened it. Read the **sign contrast between the two
model-specific spaces**: ERA5-static is the *only* space with Δ **< 0** —
TESSERA's residual is *more* static-organised than the baseline's, the
fingerprint of the input it **drops** — while lat16 has Δ **> 0**, the
embedding-organised error TESSERA can now remove. (Geography also flattens, but
that is the generic accuracy gain of the stronger model; elevation barely moves.)

**Blind probe (compact figure).** The one honest cross-model comparison of unused
signal: the baseline scored in lat16 (its blind space) vs TESSERA in ERA5-static
(its blind space) — both "surface error the model could not capture".
Baseline↔lat16 larger than TESSERA↔ERA5-static says the embedding subsumes more
of the surface signal than the static fields do; that TESSERA↔ERA5-static is
still `> 0` is the honest reminder that dropping the static fields leaves a
little static-organised error uncaptured.

**Caveat:** the two small folders (Australia n≈13, East Asia n≈88 shared
stations) give noisy structure scores that swing on a handful of stations; the
pattern is only reliable for the well-sampled regions (Europe, United States).
The averaged reference stops the *sign* from flipping between runs/sections, but
it cannot manufacture signal where n is tiny — read those cells as ≈0. The
headline mean-over-cells bar shows every region×var point so the small-n spread
is visible.

In [ ]:
# =============================================================================
# Which input space organises the residuals? — four-descriptor decomposition.
#
# Generalises §2's geo-vs-latent structure test to FOUR per-station descriptor
# spaces, scored by the identical structure score = 1 − rough/rough_random:
#     geographic      — (lat, lon), equirectangular (as §2); neither model's input
#     elevation+mTPI  — (elevation, delta_elevation, mtpi): a per-station physical
#                       descriptor COMMON to both models' inputs (+elev, +mtpi)
#     ERA5-static     — the 13 coarse ERA5 static grids bilinearly interpolated to
#                       each station: the BASELINE's surface conditioning (it sets
#                       include_static_fields=True; TESSERA drops it for the embedding)
#     TESSERA         — the 16-d TESSERA land-surface embedding: the surface
#                       conditioning of ConvCNP-with-TESSERA (baseline has none)
# The elevation/ERA5-static recipe is ported from
# scripts/analysis/norway_descriptor_spaces.py. The baseline's
# best-scoring space answers "which input space organises the residuals?".
#
# Reuses SR_LOSS / SR_METRIC / SR_KNN / SR_REGION_FOLDERS / SR_VARS and the
# _sr_select / _sr_load / _sr_latents / _sr_nn_idx / _sr_rough_idx / _sr_rough_random
# helpers from §1-2; adds per-station elevation + ERA5-static descriptor lookups
# from the shared dataset_timestamp_global. The random-neighbour reference is the
# SAME deterministic, averaged _sr_rough_random as §2, so geo/latent scores here
# match §2 exactly (a single random draw was too noisy at small n and flipped
# the score's sign between the two sections).
# =============================================================================

from scipy.interpolate import RegularGridInterpolator

# Shared dataset dir (stations.csv + per-region static grids); same tree the
# config's dataset_dir points at, resolved under $TESSERA_DATA_ROOT.
from tessera_downscaling.paths import data_root, dataset_dir, processed_dir

SR_DATASET_DIR = dataset_dir("dataset_timestamp_global")
# Folder -> region key for the per-region static grids (regions/<key>/). Built
# explicitly for the same reason SR_REGION_FOLDERS is (substring _folder_to_region
# would misroute 'australia' -> 'us').
SR_FOLDER_REGION = {
    "snapshot_14y_eu": "europe",
    "snapshot_14y_us": "us",
    "snapshot_14y_east_asia": "east_asia",
    "snapshot_14y_australia": "australia",
    "snapshot_14y_southern_africa": "southern_africa",
}
# Descriptor spaces in display order, with the norway-script colour scheme so the
# two figures stay visually consistent across the thesis.
SR_SPACES = ["geographic", "elevation+mTPI", "ERA5-static", "TESSERA"]
SR_SPACE_COLOUR = {
    "geographic": "#7f7f7f",
    "elevation+mTPI": "#ff7f0e",
    "ERA5-static": "#9467bd",
    "TESSERA": "#1f77b4",
}

_SR_STATIONS = None
_SR_ELEV_LUT = None
_SR_ERA5_LUT = {}  # region_key -> {sid: vec}


def _sr_stations():
    """The shared global station table (elevation/delta_elevation/mtpi/region)."""
    global _SR_STATIONS
    if _SR_STATIONS is None:
        s = pd.read_csv(SR_DATASET_DIR / "stations.csv")
        s["station_id"] = s["station_id"].astype(str)
        _SR_STATIONS = s
    return _SR_STATIONS


def _sr_elev_lut():
    """station_id -> (elevation, delta_elevation, mtpi)."""
    global _SR_ELEV_LUT
    if _SR_ELEV_LUT is None:
        s = _sr_stations()
        _SR_ELEV_LUT = {
            sid: (float(e), float(de), float(m))
            for sid, e, de, m in zip(
                s.station_id, s.elevation, s.delta_elevation, s.mtpi, strict=False
            )
        }
    return _SR_ELEV_LUT


def _sr_era5_lut(region_key):
    """station_id -> ERA5 static interp vector for stations in `region_key`.

    The raw region static_fields.npy (n_static, H, W) bilinearly interpolated to
    each station's (lat, lon); stations are clipped to the grid bounds first so
    there is no extrapolation. Deliberately the RAW static fields, not the CNN
    grid latent (which mixes in transient weather) — see norway_descriptor_spaces.
    Cached per region because every §3 folder maps to one region.
    """
    if region_key not in _SR_ERA5_LUT:
        rdir = SR_DATASET_DIR / "regions" / region_key
        sf = np.load(rdir / "static_fields.npy")  # (n_static, H, W)
        glat = np.load(rdir / "lats.npy")  # may be descending
        glon = np.load(rdir / "lons.npy")
        s = _sr_stations()
        s = s[s.region == region_key]
        q = np.column_stack(
            [
                np.clip(s.latitude.to_numpy(), glat.min(), glat.max()),
                np.clip(s.longitude.to_numpy(), glon.min(), glon.max()),
            ]
        )
        E = np.column_stack(
            [
                RegularGridInterpolator(
                    (glat, glon),
                    sf[c],
                    method="linear",
                    bounds_error=False,
                    fill_value=None,
                )(q)
                for c in range(sf.shape[0])
            ]
        ).astype(np.float32)
        _SR_ERA5_LUT[region_key] = {sid: E[i] for i, sid in enumerate(s.station_id)}
    return _SR_ERA5_LUT[region_key]


def _sr_zscore(rows):
    """Per-feature z-score of a list of descriptor vectors (guards zero-variance)."""
    X = np.asarray(rows, dtype=float)
    return (X - X.mean(0)) / (X.std(0) + 1e-8)


def _sr_space_decomp(folder, region, var, dist, loss, metric, knn, rows_out):
    """structure score of the baseline & TESSERA residuals in each descriptor
    space, appended as one row per (region, variable)."""
    region_key = SR_FOLDER_REGION.get(folder)
    if region_key is None:
        print(f"[skip] {region} · {var}: no region key for {folder}")
        return
    base_rows, tess_rows = _sr_select(folder, var, dist, loss)
    if base_rows.empty or tess_rows.empty:
        print(f"[skip] {region} · {var}: missing baseline/tessera run ({loss} loss)")
        return
    lut = _sr_latents(Path(tess_rows["run_dir"].iloc[0]) / "config.json")
    if lut is None:
        print(f"[skip] {region} · {var}: no latents recorded in tessera run config")
        return
    elut, slut = _sr_elev_lut(), _sr_era5_lut(region_key)
    base = _sr_load(base_rows, var, metric)
    tess = _sr_load(tess_rows, var, metric)
    shared = sorted(set(base) & set(tess) & set(lut) & set(elut) & set(slut))
    if len(shared) < knn + 2:
        print(
            f"[skip] {region} · {var}: only {len(shared)} stations with residual+descriptors"
        )
        return

    lat = np.array([base[s][0] for s in shared])
    lon = np.array([base[s][1] for s in shared])
    vb = np.array([base[s][2] for s in shared])
    vt = np.array([tess[s][2] for s in shared])
    mlat = np.radians(lat.mean())
    spaces = {
        "geographic": np.column_stack([lon * np.cos(mlat), lat]),  # equirect, as §2
        "elevation+mTPI": _sr_zscore([elut[s] for s in shared]),
        "ERA5-static": _sr_zscore([slut[s] for s in shared]),
        "TESSERA": _sr_zscore([lut[s] for s in shared]),
    }
    nn = {name: _sr_nn_idx(X, knn) for name, X in spaces.items()}

    def struct(v):
        rr = _sr_rough_random(v, knn)  # deterministic, averaged (identical to §2)
        return {name: 1 - _sr_rough_idx(v, idx) / rr for name, idx in nn.items()}

    bs, ts = struct(vb), struct(vt)
    row = dict(region=region, variable=var, n=len(shared))
    for name in SR_SPACES:
        row[f"base_{name}"] = bs[name]
        row[f"tess_{name}"] = ts[name]
    row["base_best"] = max(SR_SPACES, key=lambda k: bs[k])
    row["tess_best"] = max(SR_SPACES, key=lambda k: ts[k])
    rows_out.append(row)


SR_SPACE_ROWS = []
for _folder, _region in SR_REGION_FOLDERS.items():
    for _var, _dist, _unit in SR_VARS:
        _sr_space_decomp(
            _folder, _region, _var, _dist, SR_LOSS, SR_METRIC, SR_KNN, SR_SPACE_ROWS
        )

sr_space_df = pd.DataFrame(SR_SPACE_ROWS)
print(
    f"\nDescriptor-space {SR_METRIC} structure score ({SR_LOSS.upper()} loss, k={SR_KNN})."
)
print(
    "structure score = 1 − roughness/random_roughness  (0 = no structure, ->1 = coherent)."
)
print(
    "QUESTION: which descriptor space gives the BASELINE residuals the highest "
    "structure score (base_best)? TESSERA winning => the embedding organises the "
    "residuals better than geography or the physical descriptors."
)
_base_cols = [f"base_{s}" for s in SR_SPACES]
_tess_cols = [f"tess_{s}" for s in SR_SPACES]
display(sr_space_df[["region", "variable", "n"] + _base_cols + ["base_best"]].round(4))
display(sr_space_df[["region", "variable", "n"] + _tess_cols + ["tess_best"]].round(4))

# Which single space each model is BLIND to (never received as input); the other
# three it ingests. This flips how a bar reads (see markdown):
#   ingested space -> UNEXPLOITED residual (had the info, didn't fully remove it)
#   blind space    -> UNUSED signal (surface-organised error along an unseen axis)
SR_MODEL_BLIND = {"base": "TESSERA", "tess": "ERA5-static"}

# The per-region structure / blind-probe figures below keep only regions where the
# k=8 neighbourhood is still LOCAL (k/n < ~0.15), so the score measures local bias
# coherence rather than regional spread. Keeps Europe/US/East Asia (n>=88); drops
# Southern Africa (n~37, k/n~0.22) and Australia (n~13, k/n~0.62). Looser than the
# impartial cells (Europe+US): the structure score is a bounded in-sample statistic,
# so -- unlike the CV R^2 -- it stays interpretable down to k/n~0.1.
SR_STRUCT_MIN_N = 50


def _sr_tag(ax, tag):
    """Small panel label in the top-left corner. Titles are kept OFF the plots
    (printed as captions below) so they paste straight into the paper; the tag
    lets a caption refer to panel (a)/(b)/(c)."""
    ax.text(
        0.015,
        0.985,
        tag,
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=11,
        fontweight="bold",
    )


def _sr_caption(name, text):
    """Print a figure caption (the on-plot title, moved here to be copyable)."""
    print(f"\n[caption · {name}]\n{text}\n")


if not sr_space_df.empty:
    # ===== FIG 1 — headline: per-model "which space organises" + Delta =====
    fig, axes = plt.subplots(1, 3, figsize=(18, 4.9))
    for ax, tag, (model, cols, prefix) in zip(
        axes,
        "ab",
        [
            ("ConvCNP (without TESSERA)", _base_cols, "base"),
            ("ConvCNP with TESSERA", _tess_cols, "tess"),
        ],
        strict=False,
    ):
        means = [sr_space_df[c].mean() for c in cols]
        x = np.arange(len(SR_SPACES))
        blind = SR_MODEL_BLIND[prefix]
        bars = ax.bar(
            x,
            means,
            color=[SR_SPACE_COLOUR[s] for s in SR_SPACES],
            edgecolor="black",
            linewidth=0.6,
            zorder=2,
        )
        bars[SR_SPACES.index(blind)].set_hatch("///")  # blind space = unused signal
        for xi, col in zip(x, cols, strict=False):
            pts = sr_space_df[col].values
            jit = np.linspace(-0.18, 0.18, len(pts)) if len(pts) > 1 else [0.0]
            ax.scatter(
                xi + jit, pts, s=22, color="black", alpha=0.55, zorder=3, linewidth=0
            )
        for xi, m in zip(x, means, strict=False):
            ax.text(
                xi,
                m + 0.004,
                f"{m:.3f}",
                ha="center",
                va="bottom",
                fontsize=9,
                fontweight="bold",
            )
        ax.axhline(0, color="k", lw=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels(SR_SPACES, rotation=20, ha="right", fontsize=9)
        ax.set_ylabel(f"mean {SR_METRIC} structure score")
        ax.grid(axis="y", ls=":", alpha=0.5)
        ax.set_axisbelow(True)
        _sr_tag(ax, f"({tag})")
    # panel (c): what TESSERA absorbs -- Delta = base - tess structure per space.
    axd = axes[2]
    dmean = [
        sr_space_df[f"base_{s}"].mean() - sr_space_df[f"tess_{s}"].mean()
        for s in SR_SPACES
    ]
    x = np.arange(len(SR_SPACES))
    axd.bar(
        x,
        dmean,
        color=[SR_SPACE_COLOUR[s] for s in SR_SPACES],
        edgecolor="black",
        linewidth=0.6,
        zorder=2,
    )
    for xi, s in zip(x, SR_SPACES, strict=False):
        pts = (sr_space_df[f"base_{s}"] - sr_space_df[f"tess_{s}"]).values
        jit = np.linspace(-0.18, 0.18, len(pts)) if len(pts) > 1 else [0.0]
        axd.scatter(
            xi + jit, pts, s=22, color="black", alpha=0.55, zorder=3, linewidth=0
        )
    for xi, m in zip(x, dmean, strict=False):
        axd.text(
            xi,
            m + (0.004 if m >= 0 else -0.004),
            f"{m:+.3f}",
            ha="center",
            va="bottom" if m >= 0 else "top",
            fontsize=9,
            fontweight="bold",
        )
    axd.axhline(0, color="k", lw=0.8)
    axd.set_xticks(x)
    axd.set_xticklabels(SR_SPACES, rotation=20, ha="right", fontsize=9)
    axd.set_ylabel(f"Δ mean {SR_METRIC} structure (without − with TESSERA)")
    axd.grid(axis="y", ls=":", alpha=0.5)
    axd.set_axisbelow(True)
    _sr_tag(axd, "(c)")
    _sr_caption(
        "structure decomposition",
        "Which per-station input space organises the ConvCNP bias residuals, and what "
        "conditioning on TESSERA absorbs. Bars are the mean neighbourhood structure score "
        "(1 − roughness / random-neighbour roughness) over the ten region×variable "
        "cells; dots are the individual cells. (a) ConvCNP without TESSERA and (b) ConvCNP "
        "with TESSERA: the descriptor space each model never received as input is hatched — "
        "its score reads as unused signal (surface-organised error the model could not "
        "capture) — while the solid bars are ingested inputs (unexploited residual). "
        "(c) Δ = (without − with TESSERA) per space; positive means conditioning on "
        "TESSERA flattened that space's structure. The sign contrast between ERA5-static "
        "(Δ<0, the input TESSERA drops) and TESSERA (Δ>0, the input it gains) is the "
        "input swap.",
    )
    plt.show()

    # ===== FIG 2 — blind probe (aggregate over the ten cells) =====
    fig, ax = plt.subplots(figsize=(6.4, 4.8))
    probe = [
        (
            "ConvCNP (without TESSERA)\nblind to the TESSERA embedding",
            "base_TESSERA",
            "TESSERA",
        ),
        (
            "ConvCNP with TESSERA\nblind to ERA5-static",
            "tess_ERA5-static",
            "ERA5-static",
        ),
    ]
    xs = np.arange(len(probe))
    _tops, _bots = [], []
    for xi, (lbl, col, sp) in zip(xs, probe, strict=False):
        vals = sr_space_df[col].values
        m, sd = float(vals.mean()), float(vals.std())
        _tops.append(m + sd)
        _bots.append(m - sd)
        ax.bar(
            xi,
            m,
            yerr=sd,
            color=SR_SPACE_COLOUR[sp],
            edgecolor="black",
            linewidth=0.6,
            hatch="///",
            zorder=2,
            error_kw=dict(ecolor="black", capsize=6, lw=1.3),
        )
        ax.text(
            xi, m + sd + 0.005, f"{m:.3f}", ha="center", va="bottom", fontweight="bold"
        )
    ax.axhline(0, color="k", lw=0.8)
    ax.set_ylim(min(0.0, min(_bots)) - 0.01, max(_tops) + 0.03)
    ax.set_xticks(xs)
    ax.set_xticklabels([p[0] for p in probe], fontsize=9)
    ax.set_ylabel(f"unused {SR_METRIC} signal (structure in the blind space)")
    ax.grid(axis="y", ls=":", alpha=0.5)
    ax.set_axisbelow(True)
    _sr_caption(
        "blind probe (aggregate)",
        "Blind probe: each model scored only in the one descriptor space it never received "
        "— ConvCNP without TESSERA in the TESSERA embedding, ConvCNP with TESSERA in the "
        "ERA5-static fields. Both quantities are unused surface signal (structure in the "
        "blind space the model could not capture). Bar = mean over the ten region×variable "
        "cells; error bar = ±1 SD across those cells. The without-TESSERA model leaves "
        "more unused signal (0.048) than the with-TESSERA model (0.031), indicating the "
        "TESSERA embedding subsumes more surface signal than the ERA5 static fields; the "
        "with-TESSERA value being >0 shows dropping the static fields still leaves a little "
        "static-organised error uncaptured. The ±1 SD bars are wide — the gap is "
        "within one cross-cell SD.",
    )
    plt.show()

    # ===== FIG 3 — blind probe resolved by region x variable =====
    _probe_bars = [
        ("ConvCNP (without TESSERA) — blind to TESSERA", "base_TESSERA", "#1f77b4"),
        ("ConvCNP with TESSERA — blind to ERA5-static", "tess_ERA5-static", "#9467bd"),
    ]
    _pt = sr_space_df[[col for _, col, _ in _probe_bars]]
    _pmin = min(0.0, float(_pt.min().min()))
    _pmax = float(_pt.max().max())
    _ppad = 0.06 * (_pmax - _pmin if _pmax > _pmin else 1.0)
    fig, axes = plt.subplots(
        1, len(SR_VARS), figsize=(8.5 * len(SR_VARS), 5), squeeze=False
    )
    for ax, tag, (var, _dist, _unit) in zip(axes[0], "ab", SR_VARS, strict=False):
        sub = sr_space_df[sr_space_df["variable"] == var].set_index("region")
        regs = [
            r
            for r in SR_REGION_FOLDERS.values()
            if r in sub.index and sub.loc[r, "n"] >= SR_STRUCT_MIN_N
        ]
        x = np.arange(len(regs))
        w = 0.8 / len(_probe_bars)
        for j, (lbl, col, cc) in enumerate(_probe_bars):
            off = (j - (len(_probe_bars) - 1) / 2) * w
            ax.bar(
                x + off,
                sub.loc[regs, col].values,
                w,
                color=cc,
                edgecolor="black",
                linewidth=0.3,
                hatch="///",
                label=lbl,
            )
        ax.axhline(0, color="k", lw=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels(regs, rotation=25, ha="right", fontsize=9)
        ax.set_ylabel(f"unused {SR_METRIC} signal (structure in the blind space)")
        ax.set_ylim(_pmin - _ppad, _pmax + _ppad)
        ax.grid(axis="y", ls=":", alpha=0.5)
        ax.set_axisbelow(True)
        _sr_tag(ax, f"({tag}) {var}")
    axes[0][0].legend(fontsize=8, framealpha=0.9, loc="upper right")
    _sr_caption(
        "blind probe by region×variable",
        "Blind probe resolved by region and variable — the per-cell unused surface signal "
        "underlying the aggregate. Blue: ConvCNP without TESSERA scored in the TESSERA "
        "embedding it never received; purple: ConvCNP with TESSERA scored in the ERA5-static "
        "fields it drops. (a) t2m, (b) wind. Higher = more surface-organised error the model "
        "could not capture in its blind space; the wind panel shows the without-TESSERA "
        "model's large unused embedding signal across Europe, the US and East Asia "
        "(Southern Africa and Australia are dropped — too few stations for a local k=8 neighbourhood).",
    )
    plt.show()

    # ===== FIG 4-5 — per-model grouped bars across the four spaces, by region =====
    _all_cols = _base_cols + _tess_cols
    _ymin = min(0.0, float(sr_space_df[_all_cols].min().min()))
    _ymax = float(sr_space_df[_all_cols].max().max())
    _pad = 0.06 * (_ymax - _ymin if _ymax > _ymin else 1.0)
    _ylim = (_ymin - _pad, _ymax + _pad)

    for _model, _prefix in [
        ("ConvCNP (without TESSERA)", "base"),
        ("ConvCNP with TESSERA", "tess"),
    ]:
        blind = SR_MODEL_BLIND[_prefix]
        fig, axes = plt.subplots(
            1, len(SR_VARS), figsize=(8.5 * len(SR_VARS), 5), squeeze=False
        )
        for ax, tag, (var, _dist, _unit) in zip(axes[0], "ab", SR_VARS, strict=False):
            sub = sr_space_df[sr_space_df["variable"] == var].set_index("region")
            regs = [
                r
                for r in SR_REGION_FOLDERS.values()
                if r in sub.index and sub.loc[r, "n"] >= SR_STRUCT_MIN_N
            ]
            x = np.arange(len(regs))
            w = 0.8 / len(SR_SPACES)
            for j, space in enumerate(SR_SPACES):
                off = (j - (len(SR_SPACES) - 1) / 2) * w
                ax.bar(
                    x + off,
                    sub.loc[regs, f"{_prefix}_{space}"].values,
                    w,
                    color=SR_SPACE_COLOUR[space],
                    edgecolor="black",
                    linewidth=0.3,
                    hatch=("///" if space == blind else None),
                    label=(f"{space} (not an input)" if space == blind else space),
                )
            ax.axhline(0, color="k", lw=0.8)
            ax.set_xticks(x)
            ax.set_xticklabels(regs, rotation=25, ha="right", fontsize=9)
            ax.set_ylabel(f"{SR_METRIC} structure score (1 − rough/random)")
            ax.set_ylim(*_ylim)
            ax.grid(axis="y", ls=":", alpha=0.5)
            ax.set_axisbelow(True)
            _sr_tag(ax, f"({tag}) {var}")
        axes[0][0].legend(fontsize=8, framealpha=0.9, loc="upper right", ncol=2)
        _sr_caption(
            f"structure by space — {_model}",
            f"{_model}: neighbourhood structure score of the per-station bias across the four "
            f"descriptor spaces, by region (Europe, US, East Asia). (a) t2m, (b) wind. The hatched bar ({blind}) is the "
            f"space this model never received as input — its score is unused signal — "
            f"while the solid bars are ingested inputs (unexploited residual).",
        )
        plt.show()

### 3 (summary). Structure by space, averaged over both models — a coarse headline

Mean structure score per descriptor space, averaging the two models (per cell) and over the
ten region$\times$var cells; error bar = $\pm$1 SD across cells.

**Validity — read with the §3 caveat.** The model-average is commensurate only for the two
spaces **both** models ingest (geographic, elevation+mTPI), where both scores read as
*unexploited residual*. For **ERA5-static** and **TESSERA** (hatched, starred) it blends an
*ingested* score (unexploited residual, one model) with a *blind* score (unused signal, the
other) — different quantities — and this actually **flips the geographic$\leftrightarrow$ERA5-static
ranking** relative to the clean baseline view (TESSERA's high ERA5-static score is *unused
signal*, not error it failed to remove). So the average is a rough "where are the residuals most
coherent" summary only. The valid answer to *which space organises the errors* is the
**baseline** ranking; the per-model panels above isolate each variant's room for improvement.


In [ ]:
# -----------------------------------------------------------------------------
# 3 (summary): structure score by space, averaged over both models.
# Per space: mean over the ten region x var cells of the per-cell model-average
# (base + tess)/2; error bar = +/-1 SD across cells. ERA5-static and TESSERA are
# hatched/starred because the average there mixes an ingested (unexploited-residual)
# and a blind (unused-signal) score -- not commensurate; see the per-model panels.
# -----------------------------------------------------------------------------
_avg = {s: (sr_space_df[f"base_{s}"] + sr_space_df[f"tess_{s}"]) / 2 for s in SR_SPACES}
_amean = [_avg[s].mean() for s in SR_SPACES]
_asd = [_avg[s].std() for s in SR_SPACES]
_mixed = {"ERA5-static", "TESSERA"}  # not commensurate across models
_alabels = [f"{s}*" if s in _mixed else s for s in SR_SPACES]

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(SR_SPACES))
_bars = ax.bar(
    x,
    _amean,
    yerr=_asd,
    color=[SR_SPACE_COLOUR[s] for s in SR_SPACES],
    edgecolor="black",
    linewidth=0.6,
    zorder=2,
    capsize=5,
    error_kw=dict(lw=1.0, ecolor="0.2"),
)
for s in _mixed:
    _bars[SR_SPACES.index(s)].set_hatch("///")  # average mixes ingested + blind here
for xi, m, sd in zip(x, _amean, _asd, strict=False):
    ax.text(
        xi,
        m + sd + 0.002,
        f"{m:.3f}",
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight="bold",
    )
ax.axhline(0, color="k", lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(_alabels, rotation=20, ha="right")
ax.set_ylabel(f"mean {SR_METRIC} structure score (both models)")
ax.grid(axis="y", ls=":", alpha=0.5)
ax.set_axisbelow(True)
plt.show()

# surface the caveat numerically: the average flips geographic vs ERA5-static.
_rank = lambda key: [s for s in sorted(SR_SPACES, key=key, reverse=True)]
print(
    "which space organises the ERRORS (baseline residuals):",
    _rank(lambda s: sr_space_df[f"base_{s}"].mean()),
)
print(
    "ranking by both-model average (distorted for * spaces): ",
    _rank(lambda s: _avg[s].mean()),
)
print(
    "* average mixes an ingested (unexploited-residual) and a blind (unused-signal) score."
)

### 3b. Raw roughness $R^{\mathcal D}$ per descriptor space — without vs with TESSERA

§3 reports the magnitude-normalised **structure score**. The "the embedding smooths the
bias field" paragraph is about the **raw roughness** $R^{\mathcal D}=\tfrac1N\sum_i\big|b_i-
\overline{b}_{\,\mathcal N_{\mathcal D}(i)}\big|$ (mean absolute deviation of each station's
bias $b_i$ from its $k$ nearest neighbours in space $\mathcal D$). This cell recomputes
$R^{\mathcal D}$ for every space and both models and compares them directly.

**Caveat:** $R^{\mathcal D}$ scales with error magnitude, so a lower $R^{\mathcal D}$ partly
just reflects TESSERA's smaller errors (empirically the roughness ratio $\approx$ the
$|\text{bias}|$ ratio in every well-sampled region). The magnitude-normalised structure
score of §3 is what isolates *de-structuring* beyond shrinkage.


In [ ]:
# -----------------------------------------------------------------------------
# 3b. Raw roughness R^D per descriptor space, ConvCNP without vs with TESSERA.
# Recomputes R^D (§3's struct() keeps only 1 - R^D/R^D_random). Reuses the §3
# helpers/knobs; lower R^D = smoother bias field.
# -----------------------------------------------------------------------------
_rough_rows = []
for _folder, _region in SR_REGION_FOLDERS.items():
    _rk = SR_FOLDER_REGION[_folder]
    for _var, _dist, _unit in SR_VARS:
        _br, _tr = _sr_select(_folder, _var, _dist, SR_LOSS)
        if _br.empty or _tr.empty:
            continue
        _lut = _sr_latents(Path(_tr["run_dir"].iloc[0]) / "config.json")
        if _lut is None:
            continue
        _elut, _slut = _sr_elev_lut(), _sr_era5_lut(_rk)
        _base = _sr_load(_br, _var, SR_METRIC)
        _tess = _sr_load(_tr, _var, SR_METRIC)
        _shared = sorted(set(_base) & set(_tess) & set(_lut) & set(_elut) & set(_slut))
        if len(_shared) < SR_KNN + 2:
            continue
        _lat = np.array([_base[s][0] for s in _shared])
        _lon = np.array([_base[s][1] for s in _shared])
        _vb = np.array([_base[s][2] for s in _shared])
        _vt = np.array([_tess[s][2] for s in _shared])
        _mlat = np.radians(_lat.mean())
        _spaces = {
            "geographic": np.column_stack([_lon * np.cos(_mlat), _lat]),
            "elevation+mTPI": _sr_zscore([_elut[s] for s in _shared]),
            "ERA5-static": _sr_zscore([_slut[s] for s in _shared]),
            "TESSERA": _sr_zscore([_lut[s] for s in _shared]),
        }
        _row = dict(region=_region, variable=_var, n=len(_shared))
        for _name, _X in _spaces.items():
            _nn = _sr_nn_idx(_X, SR_KNN)
            _row[f"base_{_name}"] = _sr_rough_idx(_vb, _nn)
            _row[f"tess_{_name}"] = _sr_rough_idx(_vt, _nn)
        _rough_rows.append(_row)
sr_rough_df = pd.DataFrame(_rough_rows)
print(
    f"Raw {SR_METRIC} roughness R^D per descriptor space (k={SR_KNN}); lower = smoother bias field."
)
display(sr_rough_df.round(3))

# grouped bars: descriptor space on x, ConvCNP without vs with TESSERA; bar = mean
# over region x var, error bars = +/-1 SD across those cells (cross-region spread).
_MODELS = [
    ("base", "ConvCNP (without TESSERA)", "#3182bd"),
    ("tess", "ConvCNP with TESSERA", "#e6550d"),
]
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(SR_SPACES))
w = 0.38
for j, (pre, label, col) in enumerate(_MODELS):
    off = (j - 0.5) * w
    means = [sr_rough_df[f"{pre}_{s}"].mean() for s in SR_SPACES]
    sds = [
        sr_rough_df[f"{pre}_{s}"].std() for s in SR_SPACES
    ]  # spread across region x var cells
    ax.bar(
        x + off,
        means,
        w,
        yerr=sds,
        color=col,
        edgecolor="black",
        linewidth=0.4,
        label=label,
        zorder=2,
        capsize=4,
        error_kw=dict(lw=1.0, ecolor="0.2"),
    )
ax.set_xticks(x)
ax.set_xticklabels(SR_SPACES, rotation=15, ha="right")
ax.set_ylabel(r"bias roughness $R^{\mathcal{D}}$  (mean over region$\times$var)")
ax.grid(axis="y", ls=":", alpha=0.5)
ax.set_axisbelow(True)
ax.legend(frameon=True, framealpha=0.9)
plt.show()

### 3c. Impartial test — which space carries the raw downscaling signal?

The structure score and per-model regressions all score a *trained model's* residual, so a
space that model **ingested** is handicapped (its structure was already removed) — the
"but it saw this vs that" objection. This cell removes that confound by scoring a
**model-blind target**: the raw downscaling error $b_s = \overline{(\text{ERA5-interp} - y)}_s$
per station (time-averaged), produced by bilinear ERA5 interpolation that conditioned on
**none** of the four descriptor spaces. Every space is therefore equally "unused".

For each space we fit an **impartial regressor** — a RandomForest (scale-invariant, robust to
dimension and irrelevant features, so a 2-D and a 16-D space compete fairly) — and report the
**5-fold out-of-fold $R^2$** of predicting $b_s$ from that space's features. Identical regressor
and folds for every space. Higher = the space carries more of the downscaling signal.

The ERA5-interp per-station bias is reconstructed from each run's saved `test_predictions.npz`
(no `station_indices` of its own, so we borrow them from the matched trained baseline — targets
are verified identical — and the station lat/lon from that baseline's `test_station_errors.npz`).

**Result (see below):** on the blind target, **elevation+mTPI** and **TESSERA** carry the most
signal, and the split is by variable — elevation dominates **t2m** (lapse rate), TESSERA
dominates **wind** (land-surface/roughness) where elevation is the *weakest*. This flips
elevation from last (on the baseline residual, where the baseline had removed it) to first for
t2m, and is the impartial version of "which space organises the errors". Here $n$ is the
number of *usable* test stations (with a valid per-station bias); the bars use only Europe (898)
and US (357). East Asia has just 88 usable of its 412 test-split stations — a ~4–10× sample
disparity vs US/Europe — so it, Australia (13) and Southern Africa (37) appear in the tables but
are excluded from the bars ($n\ge100$).


In [ ]:
# -----------------------------------------------------------------------------
# 3c. Impartial "which space carries the downscaling signal": RandomForest 5-fold
# CV R^2 of each descriptor space at predicting the MODEL-BLIND target
# b_s = mean_t(ERA5-interp - obs) per station. No trained model saw any descriptor
# space, so none is handicapped -- removes the ingested/blind confound.
# -----------------------------------------------------------------------------
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_score

# 'Well-sampled' = regions with >= this many USABLE test stations (valid per-station bias):
# Europe (898) and US (357) only. East Asia has just 88 usable of its 412 test-split stations
# (~4x fewer than US, ~10x fewer than Europe) -- too large a sample disparity to pool/compare
# per-region -- and Australia (13) / Southern Africa (37) are smaller still.
SR_MIN_N = 100

_RUNS_ROOT = data_root()  # holds training_runs_<folder>/


def _sr_era5_target(folder, var):
    """station_id -> (lat, lon, mean ERA5-interp residual). ERA5-interp is
    deterministic (one seed suffices); its test_predictions.npz has no
    station_indices, so borrow them + station meta from the matched trained
    baseline (targets verified identical)."""
    root = _RUNS_ROOT / f"training_runs_{folder}"
    base_stem = SR_MODELS[var]["baseline"]
    for s in (42, 123, 456):
        ep = root / f"{var}_snap_era5_interp_baseline_seed{s}/test_predictions.npz"
        bp = root / f"{base_stem}_seed{s}/test_predictions.npz"
        bse = root / f"{base_stem}_seed{s}/test_station_errors.npz"
        if not (ep.exists() and bp.exists() and bse.exists()):
            continue
        de, db = np.load(ep), np.load(bp)
        te, tb = de[f"{var}_targets"], db[f"{var}_targets"]
        if not (len(te) == len(tb) and np.allclose(te, tb, equal_nan=True)):
            continue
        resid = (de[f"{var}_predictions"] - te).astype(float)
        sidx = db[f"{var}_station_indices"].astype(int)
        m = np.load(bse, allow_pickle=True)
        sids, lats, lons = m["station_ids"], m["station_lats"], m["station_lons"]
        nst = len(sids)
        ok = np.isfinite(resid)
        sums = np.bincount(sidx[ok], weights=resid[ok], minlength=nst)
        cnts = np.bincount(sidx[ok], minlength=nst)
        return {
            str(sids[i]): (float(lats[i]), float(lons[i]), sums[i] / cnts[i])
            for i in range(nst)
            if cnts[i] > 0
        }
    return {}


def _sr_cv_r2(X, y, seed=0):
    if len(y) < 25:
        return np.nan
    rf = RandomForestRegressor(
        n_estimators=200, min_samples_leaf=3, random_state=seed, n_jobs=-1
    )
    return float(
        np.mean(
            cross_val_score(
                rf, X, y, cv=KFold(5, shuffle=True, random_state=seed), scoring="r2"
            )
        )
    )


_imp_rows = []
for _folder, _region in SR_REGION_FOLDERS.items():
    _rk = SR_FOLDER_REGION[_folder]
    for _var, _dist, _unit in SR_VARS:
        _tgt = _sr_era5_target(_folder, _var)
        if not _tgt:
            print(f"[skip] {_region} · {_var}: no ERA5-interp target")
            continue
        _br, _tr = _sr_select(_folder, _var)
        _lut = _sr_latents(Path(_tr["run_dir"].iloc[0]) / "config.json")
        if _lut is None:
            continue
        _elut, _slut = _sr_elev_lut(), _sr_era5_lut(_rk)
        _shared = sorted(set(_tgt) & set(_lut) & set(_elut) & set(_slut))
        if len(_shared) < 25:
            continue
        _lat = np.array([_tgt[s][0] for s in _shared])
        _lon = np.array([_tgt[s][1] for s in _shared])
        _y = np.array([_tgt[s][2] for s in _shared])
        _mlat = np.radians(_lat.mean())
        _feat = {
            "geographic": np.column_stack(
                [_lon * np.cos(_mlat), _lat]
            ),  # RF is scale-invariant
            "elevation+mTPI": np.array([_elut[s] for s in _shared]),
            "ERA5-static": np.array([_slut[s] for s in _shared]),
            "TESSERA": np.array([_lut[s] for s in _shared]),
        }
        _row = dict(region=_region, variable=_var, n=len(_shared))
        for _name in SR_SPACES:
            _row[_name] = _sr_cv_r2(_feat[_name], _y)
        _imp_rows.append(_row)

sr_imp_df = pd.DataFrame(_imp_rows)
print(
    "Impartial target (mean ERA5-interp residual per station), RandomForest 5-fold CV R^2:"
)
display(sr_imp_df.round(3))
for _var, _dist, _unit in SR_VARS:
    _big = sr_imp_df[(sr_imp_df.variable == _var) & (sr_imp_df.n >= SR_MIN_N)]
    _rank = sorted(SR_SPACES, key=lambda s: -_big[s].mean())
    print(
        f"{_var}: ranking (mean CV R^2 over n>={SR_MIN_N} cells) -> "
        + ", ".join(f"{s} {_big[s].mean():+.2f}" for s in _rank)
    )

# per-variable bars: descriptor space vs CV R^2, well-sampled mean + all-cell dots.
fig, axes = plt.subplots(
    1, len(SR_VARS), figsize=(7.5 * len(SR_VARS), 5), squeeze=False
)
for ax, tag, (var, _dist, _unit) in zip(axes[0], "ab", SR_VARS, strict=False):
    sub = sr_imp_df[sr_imp_df.variable == var]
    big = sub[sub.n >= SR_MIN_N]
    x = np.arange(len(SR_SPACES))
    means = [big[s].mean() for s in SR_SPACES]
    ax.bar(
        x,
        means,
        color=[SR_SPACE_COLOUR[s] for s in SR_SPACES],
        edgecolor="black",
        linewidth=0.6,
        zorder=2,
    )
    for xi, s in zip(x, SR_SPACES, strict=False):
        pts = sub[s].values
        jit = np.linspace(-0.16, 0.16, len(pts)) if len(pts) > 1 else [0.0]
        ax.scatter(
            xi + jit, pts, s=26, color="black", alpha=0.55, zorder=3, linewidth=0
        )
    for xi, m in zip(x, means, strict=False):
        ax.text(
            xi,
            m + 0.006,
            f"{m:+.2f}",
            ha="center",
            va="bottom",
            fontsize=9,
            fontweight="bold",
        )
    ax.axhline(0, color="k", lw=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(SR_SPACES, rotation=20, ha="right", fontsize=9)
    ax.set_ylabel(r"CV $R^2$ predicting (ERA5-interp $-$ obs)")
    ax.grid(axis="y", ls=":", alpha=0.5)
    ax.set_axisbelow(True)
    _sr_tag(ax, f"({tag}) {var}")
plt.show()

### 3d. The impartial target through the roughness & structure-score lenses

For continuity with §1–§3, the same **kNN roughness** $R^{\mathcal D}$ and **structure score**
$1-R^{\mathcal D}/R^{\mathcal D}_{\mathrm{random}}$, but applied to §3c's **model-blind target**
(the per-station ERA5-interp residual) instead of a trained model's residual — so every space is
again on equal footing (no "it saw this vs that"). With a single target its random-neighbour
reference is shared across spaces, so the roughness and structure-score rankings are **identical**
(the structure score is just $R^{\mathcal D}$ rescaled). Both agree with the RandomForest CV
$R^2$ of §3c: **t2m → elevation+mTPI**, **wind → TESSERA**. These structure scores are in the
same units as the model-residual scores of the cell-8 panels, so they can be read side by side to
see exactly the confound the impartial target removes.


In [ ]:
# -----------------------------------------------------------------------------
# 3d. Impartial target (ERA5-interp residual) through the kNN roughness R^D and
# structure-score lenses (same recipe as §2/§3). Reuses _sr_era5_target from §3c.
# -----------------------------------------------------------------------------
_imp2_rows = []
for _folder, _region in SR_REGION_FOLDERS.items():
    _rk = SR_FOLDER_REGION[_folder]
    for _var, _dist, _unit in SR_VARS:
        _tgt = _sr_era5_target(_folder, _var)
        if not _tgt:
            continue
        _br, _tr = _sr_select(_folder, _var)
        _lut = _sr_latents(Path(_tr["run_dir"].iloc[0]) / "config.json")
        if _lut is None:
            continue
        _elut, _slut = _sr_elev_lut(), _sr_era5_lut(_rk)
        _shared = sorted(set(_tgt) & set(_lut) & set(_elut) & set(_slut))
        if len(_shared) < SR_KNN + 2:
            continue
        _lat = np.array([_tgt[s][0] for s in _shared])
        _lon = np.array([_tgt[s][1] for s in _shared])
        _v = np.array([_tgt[s][2] for s in _shared])
        _mlat = np.radians(_lat.mean())
        _spaces = {
            "geographic": np.column_stack([_lon * np.cos(_mlat), _lat]),
            "elevation+mTPI": _sr_zscore([_elut[s] for s in _shared]),
            "ERA5-static": _sr_zscore([_slut[s] for s in _shared]),
            "TESSERA": _sr_zscore([_lut[s] for s in _shared]),
        }
        _rr = _sr_rough_random(_v, SR_KNN)  # shared across spaces (single target)
        _row = dict(region=_region, variable=_var, n=len(_shared))
        for _name in SR_SPACES:
            _R = _sr_rough_idx(_v, _sr_nn_idx(_spaces[_name], SR_KNN))
            _row[f"rough_{_name}"] = _R
            _row[f"score_{_name}"] = 1 - _R / _rr
        _imp2_rows.append(_row)

sr_imp_knn_df = pd.DataFrame(_imp2_rows)
print(f"Impartial target (ERA5-interp residual) through the kNN lenses (k={SR_KNN}):")
display(sr_imp_knn_df.round(3))
for _var, _dist, _unit in SR_VARS:
    _big = sr_imp_knn_df[
        (sr_imp_knn_df.variable == _var) & (sr_imp_knn_df.n >= SR_MIN_N)
    ]
    _rk_s = sorted(SR_SPACES, key=lambda s: -_big[f"score_{s}"].mean())
    print(
        f"{_var}: structure-score rank -> "
        + ", ".join(f"{s} {_big[f'score_{s}'].mean():+.2f}" for s in _rk_s)
    )

# 2x2: rows = [roughness R^D (lower=better), structure score (higher=better)], cols = variable.
fig, axes = plt.subplots(2, len(SR_VARS), figsize=(7.5 * len(SR_VARS), 9))
_lenses = [
    ("rough", r"roughness $R^{\mathcal{D}}$ (lower = more organised)"),
    ("score", r"structure score (higher = more organised)"),
]
for _ri, (_pre, _ylab) in enumerate(_lenses):
    for _ci, (var, _dist, _unit) in enumerate(SR_VARS):
        ax = axes[_ri][_ci]
        sub = sr_imp_knn_df[sr_imp_knn_df.variable == var]
        big = sub[sub.n >= SR_MIN_N]
        x = np.arange(len(SR_SPACES))
        means = [big[f"{_pre}_{s}"].mean() for s in SR_SPACES]
        ax.bar(
            x,
            means,
            color=[SR_SPACE_COLOUR[s] for s in SR_SPACES],
            edgecolor="black",
            linewidth=0.6,
            zorder=2,
        )
        for xi, s in zip(x, SR_SPACES, strict=False):
            pts = sub[f"{_pre}_{s}"].values
            jit = np.linspace(-0.16, 0.16, len(pts)) if len(pts) > 1 else [0.0]
            ax.scatter(
                xi + jit, pts, s=24, color="black", alpha=0.55, zorder=3, linewidth=0
            )
        ax.axhline(0, color="k", lw=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels(SR_SPACES, rotation=20, ha="right", fontsize=9)
        ax.set_ylabel(_ylab, fontsize=10)
        ax.grid(axis="y", ls=":", alpha=0.5)
        ax.set_axisbelow(True)
        _sr_tag(ax, var)
plt.show()

### 3e. Three-lens agreement (impartial target)

The model-blind ERA5-interp target scored through all three lenses side by side — RandomForest
CV $R^2$ (§3c), kNN roughness $R^{\mathcal D}$ and structure score $S^{\mathcal D}$ (§3d) —
with the raw per-cell values tabulated below. **Bars use only Europe and United States** ($n\ge100$; 898 and 357 usable stations) with $\pm1$ SD
across them; the smaller folders (East Asia $n{=}88$, Southern Africa $n{\approx}37$, Australia
$n{\approx}13$) are in the tables but **excluded from the bars** — the sample disparity is large
(East Asia has 88 usable of its 412 test-split stations), and the two smallest flip wind's ranking
(ERA5-static spuriously overtakes TESSERA on Southern Africa's handful of stations). All three lenses agree on the well-sampled regions: **t2m → elevation+mTPI**,
**wind → TESSERA**.


In [ ]:
def _rename(df, pre):
    cols = ["region", "variable", "n"] + [f"{pre}{s}" for s in SR_SPACES]
    return df[cols].rename(columns={f"{pre}{s}": s for s in SR_SPACES})


print("Impartial target (ERA5-interp residual) — raw per-cell values, three lenses:\n")
print("CV R^2 predicting (ERA5-interp - obs):")
display(_rename(sr_imp_df, "").round(3))
print("Roughness R^D (lower = more organised):")
display(_rename(sr_imp_knn_df, "rough_").round(3))
print("Structure score S^D (higher = more organised):")
display(_rename(sr_imp_knn_df, "score_").round(3))

# 3 rows (lenses) x 2 cols (variables); bars = well-sampled mean +/-1 SD, no dots.
_LENSES = [
    (r"CV $R^2$ predicting (ERA5-interp $-$ obs)", sr_imp_df, lambda s: s)
    #    (r"roughness $R^{\mathcal{D}}$ (lower = more organised)", sr_imp_knn_df, lambda s: f"rough_{s}"),
    #    (r"structure score $S^{\mathcal{D}}$ (higher = more organised)", sr_imp_knn_df, lambda s: f"score_{s}")]
]
fig, axes = plt.subplots(
    len(_LENSES),
    len(SR_VARS),
    figsize=(7.5 * len(SR_VARS), 4.0 * len(_LENSES)),
    squeeze=False,
)
for _ri, (_ylab, _df, _colf) in enumerate(_LENSES):
    for _ci, (_var, _dist, _unit) in enumerate(SR_VARS):
        ax = axes[_ri][_ci]
        sub = _df[(_df.variable == _var) & (_df.n >= SR_MIN_N)]
        x = np.arange(len(SR_SPACES))
        means = [sub[_colf(s)].mean() for s in SR_SPACES]
        sds = [
            sub[_colf(s)].std(ddof=0) for s in SR_SPACES
        ]  # spread across well-sampled regions
        ax.bar(
            x,
            means,
            yerr=sds,
            color=[SR_SPACE_COLOUR[s] for s in SR_SPACES],
            edgecolor="black",
            linewidth=0.6,
            zorder=2,
            capsize=4,
            error_kw=dict(lw=1.0, ecolor="0.2"),
        )
        ax.axhline(0, color="k", lw=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels(SR_SPACES, rotation=20, ha="right", fontsize=9)
        ax.set_ylabel(_ylab, fontsize=9)
        ax.grid(axis="y", ls=":", alpha=0.5)
        ax.set_axisbelow(True)
        _sr_tag(ax, _var)
plt.show()

### 3g. The same impartial probe with the **extended hand-crafted descriptor**

§3c–f compare four spaces, and the only explicit *surface* descriptor among them is the
three-number topography vector $e(x^\star)=$ (elevation, $\Delta$elevation, mTPI) that the
no-TESSERA ConvCNP actually receives. That invites the sharpest objection to the headline
result: the hand-crafted side is impoverished, so a richer *physical* descriptor might
organise the residual just as well as the learned embedding.

This cell tests that on the **identical** model-blind target, probe, folds and station sets
as §3c, adding the 17-feature descriptor of the extended-descriptor ConvCNP arm
(`extra_descriptors.npy`, the input of the `*_extradesc_*` runs) split into its two halves:

| space | contents |
|---|---|
| **terrain stats (7f)** | neighbourhood elevation mean/std/min/max, slope, two directional gradients — 6.25 km radius, Copernicus GLO-30. *More topography* than $e(x^\star)$ carries. |
| **land cover (10f)** | WorldCover class fractions, ETH canopy height, SoilGrids clay/sand — 320 m radius. Surface *character*, which nothing in §3c represents explicitly. |
| **extended surface (17f)** | the two halves together — the hand-crafted descriptor as a single object. |

Splitting terrain from land cover matters: without it, a win for the 17-feature vector could
be read as "hand-crafted land cover is enough" when it was actually the extra DEM statistics
doing the work (or vice versa).

**One bar, one information type.** No space here is stacked with another. The TESSERA bar is
the embedding on its own, so a hand-crafted bar must not be the 17 features *plus* the paper's
elevation+mTPI vector — that would set a two-descriptor union against a one-descriptor
alternative and the union would win for uninteresting reasons. The `*_extradesc_*` ConvCNP arm
does receive elevation+mTPI and the 17 features together, but that is a model input, not a
descriptor space; the probe asks what each *kind* of information organises on its own.

Only `SRX_*` / `srx_*` names are written, so `sr_imp_df`, `SR_SPACES` and §3c–f are left
untouched and can be re-run in any order.

In [ ]:
# -----------------------------------------------------------------------------
# 3g. The impartial probe, widened with the EXTENDED hand-crafted descriptor.
#
# Same target (b_s = mean_t(ERA5-interp - obs) per station), same RandomForest,
# same 5-fold-over-stations CV, same station sets as 3c -- only the descriptor
# spaces change. Three spaces are added, all from extra_descriptors.npy:
#     terrain stats (7f)      DEM statistics over 6.25 km  -> MORE topography
#     land cover (10f)        WorldCover / canopy / soil    -> surface character
#     extended surface (17f)  the hand-crafted descriptor as a whole
# The terrain/cover split is keyed to the column NAMES in the sidecar json, not
# to a hard-coded index list, so it cannot silently mis-split if the builder's
# column order changes.
#
# Every bar is a SINGLE information type, so the comparison stays like-for-like:
# the TESSERA bar is the embedding alone, so a hand-crafted bar must likewise not
# be stacked with the paper's own elevation+mTPI vector. (The *_extradesc_*
# ConvCNP arm does receive elev+mTPI AND the 17 features together, but that is a
# model input, not a descriptor space -- probing the union here would compare a
# two-descriptor stack against a one-descriptor bar.)
#
# Writes only SRX_*/srx_* names: sr_imp_df and SR_SPACES are untouched.
# -----------------------------------------------------------------------------
SRX_PROC = processed_dir()
SRX_EXTRA_NPY = SRX_PROC / "extra_descriptors.npy"
SRX_EXTRA_IDS = SRX_PROC / "tessera_global/station_list_filtered.csv"

# Terrain half of the 17 features (the rest is land cover / vegetation / soil).
SRX_TERRAIN_FEATURES = {
    "elev_mean",
    "elev_std",
    "elev_min",
    "elev_max",
    "slope",
    "dz_dn",
    "dz_de",
}

SRX_SPACES = [
    "geographic",
    "elevation+mTPI",
    "terrain stats (7f)",
    "land cover (10f)",
    "extended surface (17f)",
    "ERA5-static",
    "TESSERA",
]
SRX_NEW = [
    "terrain stats (7f)",
    "land cover (10f)",
    "extended surface (17f)",
]  # not present in 3c
SRX_SPACE_COLOUR = dict(
    SR_SPACE_COLOUR,  # keep 3c's four colours
    **{
        "terrain stats (7f)": "#e377c2",
        "land cover (10f)": "#8c564b",
        "extended surface (17f)": "#c49a6c",
    },
)


def _srx_extra_lut():
    """station_id -> 17-feature extended descriptor, plus the column names.

    Row order follows station_list_filtered.csv (38,870 rows), NOT the dataset's
    stations.csv -- the two tables differ in length, so the ids must come from
    the sidecar the builder recorded rather than from the analysis station table.
    """
    if not (SRX_EXTRA_NPY.exists() and SRX_EXTRA_IDS.exists()):
        return None, []
    arr = np.load(SRX_EXTRA_NPY)
    ids = pd.read_csv(SRX_EXTRA_IDS)["station_id"].astype(str).values
    if len(ids) != len(arr):
        raise ValueError(
            f"{SRX_EXTRA_IDS.name} has {len(ids)} rows but "
            f"{SRX_EXTRA_NPY.name} has {len(arr)}"
        )
    meta = json.loads(
        SRX_EXTRA_NPY.with_name(SRX_EXTRA_NPY.stem + "_names.json").read_text()
    )
    names = meta["columns"]
    if len(names) != arr.shape[1]:
        raise ValueError(
            f"sidecar lists {len(names)} columns, array has {arr.shape[1]}"
        )
    return {s: arr[i] for i, s in enumerate(ids)}, list(names)


SRX_LUT, SRX_NAMES = _srx_extra_lut()
if SRX_LUT is None:
    print(
        f"[skip 3g] {SRX_EXTRA_NPY} not found — build it with "
        "scripts/preprocessing/build_extra_descriptors.py"
    )
else:
    _srx_ti = [i for i, c in enumerate(SRX_NAMES) if c in SRX_TERRAIN_FEATURES]
    _srx_ci = [i for i, c in enumerate(SRX_NAMES) if c not in SRX_TERRAIN_FEATURES]
    assert len(_srx_ti) == 7, (
        f"expected 7 terrain columns, found {len(_srx_ti)}: {_srx_ti}"
    )
    print(
        f"extended descriptor: {len(SRX_NAMES)} features "
        f"({len(_srx_ti)} terrain, {len(_srx_ci)} land cover/soil/vegetation) "
        f"at {len(SRX_LUT):,} stations"
    )

    _srx_rows = []
    for _folder, _region in SR_REGION_FOLDERS.items():
        _rk = SR_FOLDER_REGION[_folder]
        for _var, _dist, _unit in SR_VARS:
            _tgt = _sr_era5_target(_folder, _var)
            if not _tgt:
                print(f"[skip] {_region} · {_var}: no ERA5-interp target")
                continue
            _br, _tr = _sr_select(_folder, _var)
            if _tr.empty:
                print(f"[skip] {_region} · {_var}: no TESSERA run")
                continue
            _lut = _sr_latents(Path(_tr["run_dir"].iloc[0]) / "config.json")
            if _lut is None:
                continue
            _elut, _slut = _sr_elev_lut(), _sr_era5_lut(_rk)
            _base = set(_tgt) & set(_lut) & set(_elut) & set(_slut)  # 3c's set
            _shared = sorted(_base & set(SRX_LUT))
            if len(_shared) < 25:
                continue
            if len(_shared) != len(_base):  # keep 3c comparable
                print(
                    f"[note] {_region} · {_var}: {len(_base) - len(_shared)} of "
                    f"{len(_base)} stations lack an extended descriptor"
                )
            _lat = np.array([_tgt[s][0] for s in _shared])
            _lon = np.array([_tgt[s][1] for s in _shared])
            _y = np.array([_tgt[s][2] for s in _shared])
            _mlat = np.radians(_lat.mean())
            _Xe = np.array([_elut[s] for s in _shared])  # elevation+mTPI
            _Xx = np.array([SRX_LUT[s] for s in _shared])  # extended 17f
            _feat = {  # RF is scale-invariant
                "geographic": np.column_stack([_lon * np.cos(_mlat), _lat]),
                "elevation+mTPI": _Xe,
                "terrain stats (7f)": _Xx[:, _srx_ti],
                "land cover (10f)": _Xx[:, _srx_ci],
                "extended surface (17f)": _Xx,
                "ERA5-static": np.array([_slut[s] for s in _shared]),
                "TESSERA": np.array([_lut[s] for s in _shared]),
            }
            _row = dict(region=_region, variable=_var, n=len(_shared))
            for _name in SRX_SPACES:
                _row[_name] = _sr_cv_r2(_feat[_name], _y)
            _srx_rows.append(_row)

    srx_imp_df = pd.DataFrame(_srx_rows)
    print(
        "\nImpartial target (mean ERA5-interp residual per station), "
        "RandomForest 5-fold CV R^2 — extended descriptor spaces:"
    )
    display(srx_imp_df.round(3))

    for _var, _dist, _unit in SR_VARS:
        _big = srx_imp_df[(srx_imp_df.variable == _var) & (srx_imp_df.n >= SR_MIN_N)]
        _rank = sorted(SRX_SPACES, key=lambda s: -_big[s].mean())
        print(
            f"\n{_var}: ranking (mean CV R^2 over n>={SR_MIN_N} cells) -> "
            + ", ".join(f"{s} {_big[s].mean():+.2f}" for s in _rank)
        )
        _best_hand = max(
            (s for s in SRX_SPACES if s != "TESSERA"), key=lambda s: _big[s].mean()
        )
        print(
            f"  best hand-crafted space: {_best_hand} {_big[_best_hand].mean():+.3f} "
            f"vs TESSERA {_big['TESSERA'].mean():+.3f} "
            f"({'TESSERA' if _big['TESSERA'].mean() > _big[_best_hand].mean() else _best_hand} wins)"
        )

    # ---- bars: well-sampled mean +/-1 SD across regions, one panel per variable.
    # Same recipe as 3e; the three spaces absent from 3c sit under a shaded band
    # so the comparison with that figure is readable at a glance.
    _x = np.arange(len(SRX_SPACES))
    _newix = [SRX_SPACES.index(s) for s in SRX_NEW]
    fig, axes = plt.subplots(
        1, len(SR_VARS), figsize=(9.0 * len(SR_VARS), 5.6), squeeze=False
    )
    for ax, (_var, _dist, _unit) in zip(axes[0], SR_VARS, strict=False):
        sub = srx_imp_df[(srx_imp_df.variable == _var) & (srx_imp_df.n >= SR_MIN_N)]
        means = np.array([sub[s].mean() for s in SRX_SPACES])
        sds = np.array(
            [sub[s].std(ddof=0) for s in SRX_SPACES]
        )  # spread across regions
        ax.axvspan(min(_newix) - 0.5, max(_newix) + 0.5, color="0.92", zorder=0)
        ax.bar(
            _x,
            means,
            yerr=sds,
            color=[SRX_SPACE_COLOUR[s] for s in SRX_SPACES],
            edgecolor="black",
            linewidth=0.6,
            zorder=2,
            capsize=4,
            error_kw=dict(lw=1.0, ecolor="0.2"),
        )
        # Headroom is set explicitly rather than left to autoscale: the value
        # labels sit above the error caps, so autoscale (which only sees the
        # bars) puts them through the top spine and the band caption.
        _lo = min(0.0, float((means - sds).min()))
        _hi = float((means + sds).max())
        _span = max(_hi - _lo, 1e-6)
        ax.set_ylim(_lo - 0.05 * _span, _hi + 0.22 * _span)
        # x in data coords, y in axes fraction -- so the band label tracks the
        # final ylim instead of the default one bars have not yet expanded.
        ax.text(
            (min(_newix) + max(_newix)) / 2,
            0.99,
            "extended descriptor (new)",
            transform=ax.get_xaxis_transform(),
            ha="center",
            va="top",
            fontsize=8,
            color="0.35",
        )
        for xi, m, sd in zip(_x, means, sds, strict=False):
            ax.text(
                xi,
                m + sd + 0.02 * _span,
                f"{m:+.2f}",
                ha="center",
                va="bottom",
                fontsize=9,
                fontweight="bold",
            )
        ax.axhline(0, color="k", lw=0.8)
        ax.set_xticks(_x)
        ax.set_xticklabels(SRX_SPACES, rotation=25, ha="right", fontsize=9)
        ax.set_ylabel(r"CV $R^2$ predicting (ERA5-interp $-$ obs)", fontsize=9)
        ax.grid(axis="y", ls=":", alpha=0.5)
        ax.set_axisbelow(True)
        _sr_tag(ax, _var)
    plt.tight_layout()
    plt.show()

    _sr_caption(
        "3g",
        (
            "Which per-station descriptor space makes the persistent ERA5-interpolation "
            f"residual predictable at held-out stations. RandomForest 5-fold CV $R^2$, mean over "
            f"the well-sampled regions (n>={SR_MIN_N}: Europe, United States); error bar = "
            "$\\pm$1 SD across them. The three shaded spaces come from the 17-feature extended "
            "hand-crafted descriptor; the rest are as in 3c. Every bar is one descriptor space on "
            "its own -- none is stacked with another -- so the hand-crafted and learned surface "
            "descriptors are compared like for like. The target is model-blind, so no space is "
            "handicapped by having been ingested."
        ),
    )

### 3f. Per-region decomposition (impartial target)

The §3c–e bars average over the well-sampled regions; here the model-blind target is broken out
**per region** across the four descriptor spaces — the same by-region view as the per-model
figures in §3 (cell 8), but on the impartial target — restricted to the well-sampled regions
($n\ge$`SR_MIN_N`: Europe and US only; $n$ annotated under each). The variable-level story is
consistent across them: **t2m organised by elevation+mTPI, wind by TESSERA**. East Asia (88
usable stations), Australia and Southern Africa are dropped — too large a sample disparity vs
Europe/US to compare per-region; Australia additionally has no CV $R^2$ ($n<25$ makes the 5-fold
out-of-fold $R^2$ undefined).


In [ ]:
# -----------------------------------------------------------------------------
# 3f. Per-region decomposition of the impartial target (mirrors the per-model
# by-region figures of §3, cell 8): each region's four descriptor-space scores,
# for the CV R^2 and structure-score lenses. Restricted to the well-sampled
# regions (n >= SR_MIN_N: Europe/US only); n annotated per region.
# -----------------------------------------------------------------------------
_PR_LENSES = [
    (r"CV $R^2$ predicting (ERA5-interp $-$ obs)", sr_imp_df, lambda s: s),
    (
        r"structure score $S^{\mathcal{D}}$ (higher = more organised)",
        sr_imp_knn_df,
        lambda s: f"score_{s}",
    ),
]
fig, axes = plt.subplots(
    len(_PR_LENSES),
    len(SR_VARS),
    figsize=(8.5 * len(SR_VARS), 4.6 * len(_PR_LENSES)),
    squeeze=False,
)
for _ri, (_ylab, _df, _colf) in enumerate(_PR_LENSES):
    for _ci, (_var, _dist, _unit) in enumerate(SR_VARS):
        ax = axes[_ri][_ci]
        sub = _df[(_df.variable == _var) & (_df.n >= SR_MIN_N)].set_index("region")
        regs = [r for r in SR_REGION_FOLDERS.values() if r in sub.index]
        x = np.arange(len(regs))
        w = 0.8 / len(SR_SPACES)
        for _j, _space in enumerate(SR_SPACES):
            off = (_j - (len(SR_SPACES) - 1) / 2) * w
            ax.bar(
                x + off,
                sub.loc[regs, _colf(_space)].values,
                w,
                color=SR_SPACE_COLOUR[_space],
                edgecolor="black",
                linewidth=0.3,
                label=_space,
            )
        ax.axhline(0, color="k", lw=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels(
            [f"{r}\n(n={int(sub.loc[r, 'n'])})" for r in regs], fontsize=8
        )
        ax.set_ylabel(_ylab, fontsize=9)
        ax.grid(axis="y", ls=":", alpha=0.5)
        ax.set_axisbelow(True)
        _sr_tag(ax, _var)
axes[0][0].legend(fontsize=8, framealpha=0.9, loc="best", ncol=2)
plt.show()

## 4. Is the added structure aimed at the baseline's error?

The maps (§1) show that TESSERA adds fine-scale texture. This section tests whether that
texture is **skilful** rather than cosmetic, at the level of the point (median) prediction
the maps display. Decompose each TESSERA prediction into the baseline plus an increment,

$$\hat y_{\mathrm{T}}(x_s)=\hat y_b(x_s)+\Delta_s,\qquad
  \Delta_s=\hat y_{\mathrm{T}}(x_s)-\hat y_b(x_s),\qquad
  e_s=y_s-\hat y_b(x_s),$$

with per-observation gain $g_s=|e_s|-|e_s-\Delta_s|$ (so $|g_s|\le|\Delta_s|$; the increment
only *reduces* error when directed along it). The structure is skilful iff $\Delta$ aligns
with $e$: we regress $e_s=\beta\Delta_s+\eta_s$ and report
$R^2=\mathrm{corr}(\Delta,e)^2$ (baseline-error variance the increment explains) and the
directional hit-rate $H=\tfrac1N\sum_s\mathbb 1[\operatorname{sign}\Delta_s=\operatorname{sign}e_s]$.

Run over the **full test set of every region** with the main-results models
(Gaussian+mTPI t2m, truncated-normal+mTPI wind), MAE@median point estimate — reusing
`scripts/maps/residual_alignment_fulltest.py` so the notebook and the paper numbers come
from one implementation.


In [ ]:
# -----------------------------------------------------------------------------
# 4a. Per-region full-test residual alignment (MAE + CRPS + R^2 + hit-rate).
# -----------------------------------------------------------------------------
sys.path.insert(0, str(Path.cwd().parent / "scripts" / "maps"))
import residual_alignment_fulltest as raf

importlib.reload(raf)
from scipy.stats import pearsonr

_rows, _pool = [], {}
for reg in raf.REGIONS:
    for var in ["t2m", "wind"]:
        rb = raf.ens_point(reg, raf.STEM[var][0], var)
        # TESSERA arm follows the top selector (generation folder + stem).
        rt = raf.ens_point(reg, TESS_STEM[var], var, suffix=TESS_FOLDER_SUFFIX)
        if rb is None or rt is None:
            continue
        yb, tb = rb
        yt, tt = rt
        if not (len(tb) == len(tt) and np.allclose(tb, tt)):
            print(f"[skip misaligned] {reg} {var}")
            continue
        y = tb
        e = y - yb
        d = yt - yb
        r = pearsonr(d, e).statistic
        _rows.append(
            dict(
                region=reg,
                var=var,
                n=len(e),
                mae_base=np.abs(e).mean(),
                mae_tess=np.abs(e - d).mean(),
                crps_base=raf.crps_seedmean(reg, raf.STEM[var][0], var),
                crps_tess=raf.crps_seedmean(
                    reg, TESS_STEM[var], var, suffix=TESS_FOLDER_SUFFIX
                ),
                R2=r**2,
                hit_pct=100 * np.mean(np.sign(d) == np.sign(e)),
            )
        )
        _pool.setdefault(var, []).append((e, d))

align = pd.DataFrame(_rows)
align["dMAE%"] = 100 * (align.mae_base - align.mae_tess) / align.mae_base
align["dCRPS%"] = 100 * (align.crps_base - align.crps_tess) / align.crps_base
display(align.round(3))

for var in ["t2m", "wind"]:
    E = np.concatenate([x[0] for x in _pool[var]])
    D = np.concatenate([x[1] for x in _pool[var]])
    r = pearsonr(D, E).statistic
    print(
        f"pooled {var}: N={len(E):,}  R2={r**2:.2f}  hit={100 * np.mean(np.sign(D) == np.sign(E)):.1f}%"
    )

In [ ]:
# -----------------------------------------------------------------------------
# 4b. Alignment scales with the added-structure magnitude |Delta| (footnote evidence).
# Stratify every test obs by |Delta| (station-level analogue of map texture): where
# TESSERA adds little it is undirected (hit ~50%) but negligible; where it adds a lot
# -- the regime the dense maps depict -- it is reliably aimed at the baseline error
# (hit -> ~76%, within-decile R^2 -> ~0.3). This is why the modest full-test R^2 does
# not undercut the map-scale claim: it is diluted by near-zero increments.
# -----------------------------------------------------------------------------
def _pooled_ed(var):
    E, D = [], []
    for reg in raf.REGIONS:
        rb = raf.ens_point(reg, raf.STEM[var][0], var)
        # TESSERA arm follows the top selector (generation folder + stem).
        rt = raf.ens_point(reg, TESS_STEM[var], var, suffix=TESS_FOLDER_SUFFIX)
        if rb is None or rt is None:
            continue
        yb, tb = rb
        yt, tt = rt
        if len(tb) == len(tt) and np.allclose(tb, tt):
            E.append(tb - yb)
            D.append(yt - yb)
    return np.concatenate(E), np.concatenate(D)


fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
strat = {}
for ax, var in zip(axes, ["t2m", "wind"], strict=False):
    e, d = _pooled_ed(var)
    g = np.abs(e) - np.abs(e - d)
    ad = np.abs(d)
    qs = np.quantile(ad, np.linspace(0, 1, 11))
    mid, hit, imp, r2 = [], [], [], []
    for i in range(10):
        m = (ad >= qs[i]) & (ad <= qs[i + 1] if i == 9 else ad < qs[i + 1])
        mid.append(ad[m].mean())
        hit.append(100 * np.mean(np.sign(d[m]) == np.sign(e[m])))
        imp.append(100 * np.mean(g[m] > 0))
        r2.append(pearsonr(d[m], e[m]).statistic ** 2 if m.sum() > 2 else np.nan)
    strat[var] = pd.DataFrame(
        dict(mean_absDelta=mid, hit_pct=hit, pct_improved=imp, R2_within=r2)
    )
    ax.plot(mid, hit, "o-", label="directional hit-rate (%)")
    ax.plot(mid, imp, "s--", label="% improved")
    ax.axhline(50, color="0.6", lw=0.8)
    ax.set_xlabel(r"mean $|\Delta|$ in decile")
    ax.set_ylabel("%")
    ax.set_title(var)
    ax.set_ylim(45, 90)
    axr = ax.twinx()
    axr.plot(mid, r2, "^:", color="crimson")
    axr.set_ylabel(r"within-decile $R^2$", color="crimson")
    axr.set_ylim(0, None)
    ax.legend(loc="upper left", fontsize=9)
fig.suptitle(
    r"Alignment strengthens with the added-structure magnitude $|\Delta|$"
    "  (full test set, all regions)"
)
plt.show()
print("wind stratification by |Delta| decile:")
display(strat["wind"].round(2))